# Feature Mathematics: Rotation Sensitivity in cp_measure

This notebook derives WHY each sensitive cp_measure feature is rotation-dependent,
and identifies the feasible correction for each.

**Companion notebooks:**
- `analyze_rotation_sweep.ipynb` — empirical RSI measurements (run first)
- `feature_corrections.ipynb` — proof-of-concept post-hoc corrections

## The core question

For a 2D image $I(x, y)$, a counter-clockwise rotation by $\alpha$ produces:

$$I_\alpha(x, y) = I(x\cos\alpha + y\sin\alpha,\; -x\sin\alpha + y\cos\alpha)$$

We ask: which features extracted from $I_\alpha$ differ from those extracted from $I$?
The **RSI** (Rotation Sensitivity Index) from `analyze_rotation_sweep.ipynb` quantifies
this empirically.  This notebook provides the **mathematical explanation**.


## 1 — Central Moments (RSI up to 10⁶)

**Definition:**
$$\mu_{pq} = \iint (x-\bar x)^p (y-\bar y)^q\, I(x,y)\, dx\, dy$$

cp_measure stores `CentralMoment_p_q` for $p, q \in \{0,1,2,3\}$.

---
**Rotation transformation (binomial expansion):**

$$\mu'_{pq} = \sum_{a=0}^{p}\sum_{b=0}^{q}
\binom{p}{a}\binom{q}{b}(-1)^b\,
\cos^{a+q-b}(\alpha)\,\sin^{p-a+b}(\alpha)\;
\mu_{a+b,\; p+q-a-b}$$

For order-2 moments this gives the familiar rotation-of-covariance formula:
$$\mu'_{11} = (\cos^2\alpha - \sin^2\alpha)\,\mu_{11} + \cos\alpha\sin\alpha\,(\mu_{02}-\mu_{20})$$

$\mu_{11}$ is zero for an axis-aligned ellipse, maximum at 45°.  This drives `CentralMoment_1_1` RSI $\approx 10^6$.

---
**Correction strategy:** Apply the *inverse* rotation $(-\alpha)$ using the same formula.
- In simulation: use `angle_deg` directly.
- In production: use the measured `Orientation` feature ($\Delta$Orientation $= \alpha$ when the image rotates).
- **Best practice:** pre-rotate to canonical orientation before calling cp_measure.


In [1]:
from math import comb

import numpy as np


def rotate_central_moments(moments: dict, alpha: float) -> dict:
    """Rotate central moments {(p,q): value} by angle alpha (radians)."""
    c, s = np.cos(alpha), np.sin(alpha)
    result = {}
    for p, q in moments:
        val = 0.0
        for a in range(p + 1):
            for b in range(q + 1):
                coeff = (
                    comb(p, a)
                    * comb(q, b)
                    * ((-1) ** b)
                    * (c ** (a + q - b))
                    * (s ** (p - a + b))
                )
                val += coeff * moments.get((a + b, p + q - a - b), 0.0)
        result[(p, q)] = val
    return result


# Example: 90-degree rotation of an axis-aligned ellipse (mu_11 = 0 initially)
original = {
    (0, 0): 1.0,
    (2, 0): 4.0,
    (1, 1): 0.0,
    (0, 2): 1.0,
    (3, 0): 0.0,
    (2, 1): 0.0,
    (1, 2): 0.0,
    (0, 3): 0.0,
}
rot90 = rotate_central_moments(original, np.deg2rad(90))
back = rotate_central_moments(rot90, np.deg2rad(-90))

print(
    f"Original  mu_20={original[(2, 0)]:.2f}, mu_11={original[(1, 1)]:.2f}, mu_02={original[(0, 2)]:.2f}"  # noqa: E501
)
print(
    f"After 90° mu_20={rot90[(2, 0)]:.2f}, mu_11={rot90[(1, 1)]:.2f}, mu_02={rot90[(0, 2)]:.2f}  (mu_20 and mu_02 swap)"  # noqa: E501
)
print(
    f"Corrected mu_20={back[(2, 0)]:.2f}, mu_11={back[(1, 1)]:.2f}, mu_02={back[(0, 2)]:.2f}  (recovered)"  # noqa: E501
)

Original  mu_20=4.00, mu_11=0.00, mu_02=1.00
After 90° mu_20=1.00, mu_11=-0.00, mu_02=4.00  (mu_20 and mu_02 swap)
Corrected mu_20=4.00, mu_11=-0.00, mu_02=1.00  (recovered)


## 2 — Normalized Moments (same math as Central Moments)

**Definition:**
$$\eta_{pq} = \frac{\mu_{pq}}{\mu_{00}^{(p+q)/2+1}}$$

Because $\mu_{00}$ (area) is rotation-invariant, the rotation transformation is **identical**
to the central moment formula above.  `rotate_central_moments` applies directly to
normalised moment values without modification.

cp_measure stores `NormalizedMoment_p_q` for $p, q \in \{0,1,2,3\}$.


## 3 — Spatial Moments (RSI 0.08 – 6.7)

**Definition:** $M_{pq} = \iint x^p y^q\, I(x,y)\, dx\, dy$ -- computed in the **image frame**.

**Why sensitive:** Spatial moments are affected by both translation (bounding-box shift) and rotation.

**Relationship to central moments:**
$$\mu_{pq} = \sum_{a=0}^{p}\sum_{b=0}^{q}(-1)^{p-a+q-b}\binom{p}{a}\binom{q}{b}\,\bar x^{p-a}\,\bar y^{q-b}\,M_{ab}$$

**Two-step correction:**
1. Convert $M_{pq} \to \mu_{pq}$ using centroid $\bar x = M_{10}/M_{00}$, $\bar y = M_{01}/M_{00}$.
2. Apply `rotate_central_moments` to the resulting $\mu_{pq}$.

Note: $M_{00}$ (total intensity), $M_{10}/M_{00}$, $M_{01}/M_{00}$ (centroid coordinates) are invariant.


## 4 — Inertia Tensor

**Definition:**
$$\mathbf I = \begin{pmatrix}\mu_{02} & -\mu_{11} \\ -\mu_{11} & \mu_{20}\end{pmatrix}$$

cp_measure stores `InertiaTensor_i_j` (0-indexed rows/cols).

**Rotation behaviour:** As a rank-2 symmetric tensor, $\mathbf I' = R(\alpha)\,\mathbf I\,R(\alpha)^\top$.

| Element | Status |
|---|---|
| `InertiaTensor_0_1` = `InertiaTensor_1_0` = $-\mu_{11}$ | Rotation-sensitive (same as `CentralMoment_1_1`) |
| `InertiaTensor_0_0` = $\mu_{02}$, `InertiaTensor_1_1` = $\mu_{20}$ | Swap under 90° rotation |
| `InertiaTensorEigenvalues_0/1` | **Invariant** (eigenvalues are rotation-invariant) |

**Correction:** The off-diagonal $I_{01}$ corrects as $-\mu_{11}^{\text{corr}}$.
The diagonals correct through the full 2nd-order moment rotation applied to ($\mu_{02}, \mu_{11}, \mu_{20}$).


## 5 — Hu Moments: why apparent RSI is a test-design artefact

The 7 Hu invariant moments ($\phi_1\ldots\phi_7$) are polynomial combinations of $\eta_{pq}$
specifically designed to be rotation-invariant (Hu 1962).

From our data: `HuMoment_0` (RSI 0.0002) and `HuMoment_1` (RSI 0.0002) confirm invariance.
But `HuMoment_3` (RSI 16.5) and `HuMoment_4` (RSI 794) appear sensitive.

**The explanation:** $\phi_3$--$\phi_7$ all involve **third-order** terms: $\eta_{30}, \eta_{03}, \eta_{21}, \eta_{12}$.

Our synthetic cells are **axis-aligned ellipses** -- shapes with two mirror symmetry planes.
Any shape with bilateral symmetry has all odd-order cross-terms equal to zero:
$$\eta_{30} = \eta_{03} = \eta_{21} = \eta_{12} = 0 \quad (\text{for symmetric shapes})$$

Therefore $\phi_3 = \phi_4 = \phi_5 = \phi_6 = 0$ for every cell in our dataset.
Between-cell variance $\approx$ 0, making RSI = within-noise / near-zero = artificially inflated.

**Conclusion:** Hu moments ARE rotation-invariant.  The high RSI values are a **statistical artefact**
of testing on symmetric shapes, not a feature defect.  No correction is needed.


## 6 — Zernike Shape Features (`Zernike_n_m`): same artefact

`Zernike_n_m` = magnitude of the projection of the **binary mask** onto Zernike polynomial $(n, m)$:
$$\text{Zernike}_{n,m} = \frac{\sqrt{\text{Re}(c_{nm})^2 + \text{Im}(c_{nm})^2}}{\pi r_{\text{enc}}^2}$$

Under rotation: $c'_{nm} = e^{-im\alpha} c_{nm}$, so $|c'_{nm}| = |c_{nm}|$.  **Magnitudes are invariant.**

**Why odd-$m$ features show RSI up to 4 in our sweep:**
- Our ellipses have **C2 symmetry** (invariant under 180° rotation).
- A shape with $C_k$ symmetry has $c_{nm} = 0$ for all $m$ not divisible by $k$.
- For $k=2$: $c_{nm} = 0$ for all **odd** $m$.
- Consequently, every cell has $\text{Zernike}_{n,\text{odd}} \approx 0$, and RSI = noise/near-zero.

Same artefact as Hu moments.  **No correction needed** for real (asymmetric) cells.


## 7 — RadialDistribution ZernikePhase (RSI 0.06 – 4)

**Definition:** cp_measure projects the **intensity image** onto Zernike polynomials:
$$c_{nm} = \sum_{\text{pixels}} I(x,y) \cdot Z^m_n(x,y)$$
$$\text{ZernikePhase}_{nm} = \arg(c_{nm}) = \text{arctan2}(\text{Re}(c_{nm}),\;\text{Im}(c_{nm}))$$

(Note: cp_measure uses `arctan2(Re, Im)`, non-standard but self-consistent.)

**Under rotation by $\alpha$:** $c'_{nm} = e^{-im\alpha}\,c_{nm}$, so the phase shifts by $-m\alpha$.

**Correction:**
$$\phi^{\text{corr}}_{nm} = \phi_{nm} + m \cdot \alpha$$

where $\alpha$ = rotation angle in radians.  Equivalently (in production):
$$\phi^{\text{corr}}_{nm} = \phi_{nm} + m \cdot \text{Orientation}$$

This expresses the phase relative to the cell's major axis, removing the image-frame orientation dependence.
For $m = 0$: the phase is undefined (coefficient is real); `ZernikePhase_n_0` features are constant.

**RadialDistribution ZernikeMagnitude:** Should be invariant but shows RSI 0.01--0.93 from pixel
interpolation during rotation of the discrete image.  No analytical post-hoc fix; mitigate by oversampling.


## 8 — Location Features (RSI 0.6 – 5.3)

cp_measure reports the $(x, y)$ position of the **intensity-weighted centroid** and **brightest pixel**
in image coordinates.  These change with both rotation and the position of the cell in the frame.

**Correction:** Convert to polar coordinates relative to the object centroid and major axis:

$$d = \sqrt{(X - \bar x)^2 + (Y - \bar y)^2} \quad\text{(distance -- fully invariant)}$$

$$\theta_{\text{rel}} = \text{arctan2}(Y - \bar y,\; X - \bar x) - \text{Orientation} \quad\text{(angle relative to major axis)}$$

The distance $d$ is completely rotation-invariant.  The relative angle $\theta_{\text{rel}}$ is invariant
up to the 180° ambiguity of `Orientation`.

**Recommendation:** add `Location_CenterMassDistance` and `Location_MaxIntensityDistance`
as new rotation-invariant features in cp_measure.


## 9 — Granularity (RSI 0.05 – 49, not analytically correctable)

**Definition:** $G_k = (A_{k-1} - A_k) / A_0$ where $A_k$ = image area after morphological opening
with a disk SE of radius $k$ pixels.

A **continuous** circular SE would give rotation-invariant results.  The discrete approximation
introduces directional bias: the digitised disk has slightly more pixels along axis-aligned directions,
causing $A_k$ to vary slightly with image orientation.

**No analytical post-hoc correction exists.**  Mitigation requires measurement-stage changes:
- Pre-rotate the image to canonical orientation before measurement.
- Use subpixel-accurate structuring elements.


## 10 — Summary and implementation guide

In [2]:
import pandas as pd

summary = pd.DataFrame(
    [
        (
            "CentralMoment_p_q",
            "0.1 -- 10^6",
            "Binomial rotation inverse",
            "post-hoc OK",
        ),
        ("NormalizedMoment_p_q", "0.6 -- 3e5", "Same as CentralMoment", "post-hoc OK"),
        (
            "SpatialMoment_p_q",
            "0.08 -- 6.7",
            "Translate to centroid, then rotate",
            "post-hoc OK",
        ),
        ("InertiaTensor off-diag", "~4e5", "= -mu11 corrected", "post-hoc OK"),
        (
            "InertiaTensor diagonal",
            "0.19",
            "Rotate (mu02, mu11, mu20) tensor",
            "post-hoc OK",
        ),
        ("InertiaTensorEigenvalues", "< 0.01", "Already invariant", "--"),
        ("HuMoment_0, _1", "< 0.001", "Already invariant", "--"),
        (
            "HuMoment_2 -- _6",
            "0.75 -- 794",
            "Artefact of symmetric test shapes",
            "No fix needed",
        ),
        (
            "Zernike_n_odd-m",
            "0.5 -- 4",
            "Artefact of C2 symmetric test shapes",
            "No fix needed",
        ),
        ("Zernike_n_even-m", "< 0.09", "Already invariant (pixel noise only)", "--"),
        (
            "RadialDistribution ZernikePhase_n_m",
            "0.06 -- 4",
            "phase + m * Orientation",
            "post-hoc OK",
        ),
        (
            "RadialDistribution ZernikeMag_n_m",
            "0.01 -- 0.93",
            "Pixel interp; no analytical fix",
            "pre-orient",
        ),
        ("Granularity_k", "0.05 -- 49", "Discrete SE; no analytical fix", "pre-orient"),
        (
            "Location CenterMass/MaxInt X/Y",
            "0.6 -- 5.3",
            "Convert to distance from centroid",
            "post-hoc OK",
        ),
        (
            "Intensity edge (min, std)",
            "0.04 -- 0.15",
            "Edge pixel set changes; no simple fix",
            "pre-orient",
        ),
        ("Texture (Haralick)", "< 0.05", "Already invariant (4-dir GLCM)", "--"),
        ("Shape (Area, Eccentricity...)", "< 0.01", "Already invariant", "--"),
        ("Feret diameters", "< 0.001", "Already invariant", "--"),
    ],
    columns=["Feature/pattern", "RSI (our sweep)", "Correction", "Where to implement"],
)

pd.set_option("display.max_colwidth", 55)
print(summary.to_string(index=False))

                    Feature/pattern RSI (our sweep)                            Correction Where to implement
                  CentralMoment_p_q     0.1 -- 10^6             Binomial rotation inverse        post-hoc OK
               NormalizedMoment_p_q      0.6 -- 3e5                 Same as CentralMoment        post-hoc OK
                  SpatialMoment_p_q     0.08 -- 6.7    Translate to centroid, then rotate        post-hoc OK
             InertiaTensor off-diag            ~4e5                     = -mu11 corrected        post-hoc OK
             InertiaTensor diagonal            0.19      Rotate (mu02, mu11, mu20) tensor        post-hoc OK
           InertiaTensorEigenvalues          < 0.01                     Already invariant                 --
                     HuMoment_0, _1         < 0.001                     Already invariant                 --
                   HuMoment_2 -- _6     0.75 -- 794     Artefact of symmetric test shapes      No fix needed
                   

## References

- Hu, M.-K. (1962). Visual pattern recognition by moment invariants. *IRE Trans. Inf. Theory* 8(2), 179--187.
- Teague, M. R. (1980). Image analysis via the general theory of moments. *J. Opt. Soc. Am.* 70(8), 920--930.
- Boland et al. (1998). Automated recognition of patterns in fluorescence microscopy. *Cytometry* 33(3), 366--375.
- centrosome source: `zernike.py` -- uses arctan2(Re, Im) convention for phase.
